# [title] One-state Model: a posterior network trained with BayesFlow

**Question.** Can a neural network, trained once on simulated Sittings of the one-state
Model, give a trustworthy posterior of the Model's four parameters for a new Sitting, at
its full length, without downsampling and without padding?

**Scope.** Step C3b of [issue #12](https://github.com/opherdonchin/MotorAdpatationSBI/issues/12).
Background: the wiki pages
[Concept-amortized-inference](https://github.com/opherdonchin/MotorAdpatationSBI/wiki/Concept-amortized-inference)
(what is trained, and how it is checked),
[Concept-summary-networks](https://github.com/opherdonchin/MotorAdpatationSBI/wiki/Concept-summary-networks)
(how the network reads a Sitting) and
[Concept-bayesflow-loss-functions](https://github.com/opherdonchin/MotorAdpatationSBI/wiki/Concept-bayesflow-loss-functions)
(what the loss is). Why this summary network and this way of batching:
[decision 0007](../docs/decisions/0007-summary-network-for-sittings.md). The Model,
Prior and Schedule Design: [docs/models/one_state.md](../docs/models/one_state.md).
Terms: [glossary](../docs/glossary.md).

**What is built.** An Inference Engine: a summary network (convolution, then
recurrence) that reads every Trial of a Sitting ($p_t$, $v_t$, $y_t$), followed by a
flow-matching inference network that turns its 12 summary numbers into draws from the
posterior of $A$, $B$, $\sigma_\eta$ and $\sigma_\varepsilon$. It is trained on Sittings
simulated as they are needed, with the Prior and Schedule Design of decision 0005, and
checked on a saved test Experiment.

**The one rule that makes this work without padding:** all Sittings in a training batch
share one Schedule, so they have one length. Different batches have different Schedules
and lengths. The trained network takes a Sitting of any length as it is.

**Checks:**

1. *Training converged:* the loss on held-out Sittings has flattened and has not pulled
   away from the training loss.
2. *Calibration* (the Check that decides whether to trust the Engine): over the test
   Sittings, the true value of each parameter falls in the posterior's intervals as
   often as it should.
3. *Recovery and contraction* (reported, not pass or fail): how close the posterior's
   centre is to the true value, and how much narrower the posterior is than the Prior.
4. *Sitting length:* Checks 2 and 3 separately for the shorter and the longer test
   Sittings.

**The steps, and what passes between them.** $n$ is a number of Sittings and $T$ a
number of Trials.

| Step | Takes | Gives |
|---|---|---|
| 1. Simulate a batch | the Prior, the Schedule Design, $n$ | arrays `p`, `v`, `y` of shape $(n, T)$; `A`, `B`, `sigma_eta`, `sigma_epsilon` of shape $(n, 1)$ |
| 2. Route the arrays (the adapter) | a batch | what the summary network reads, $(n, T, 3)$; what the inference network learns, $(n, 4)$ |
| 3. Build the Inference Engine | sizes of the two networks | an untrained Engine |
| 4. Train | the Engine, and a fresh batch for every training step | the trained Engine, saved; the loss after every epoch |
| 5. Check 1 | the losses | has training converged? |
| 6. Make the test Experiment | the Prior, the Schedule Design | 300 Sittings and their Ground Truth, saved |
| 7. Draw from the posteriors | the trained Engine, the test Sittings | 1,000 draws of each parameter for each Sitting: $(300, 1000, 1)$ |
| 8. Checks 2 to 4 | the draws, the Ground Truth | calibration, recovery, contraction, by Sitting length |
| 9. Use the Engine on one Sitting | the draws for one Sitting | a picture of its posterior |

Before the steps come three cells of setup: imports, every setting in one place, and the
random number streams.

## [setup] Setup

Every number that can be changed is in `#[config]`; nothing further down sets one.

In [ ]:
# [imports]
import json
import pathlib
import subprocess

import bayesflow as bf
import keras
import numpy as np
import pandas as pd
from scipy import special, stats

from designs import visuomotor_adaptation_experiment as des
from models import one_state as mdl
from motor_sbi import experiments

In [ ]:
# [config] Root seed, provenance, Prior and Schedule Design, sizes, networks, training
root_seed = 112907950985685676175328411057333210321  # secrets.randbits(128)

# Provenance: the repository's top folder, the commit that produced the outputs (with
# -dirty if there were uncommitted changes), and where this notebook's outputs go.
REPO = pathlib.Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
GIT_COMMIT = subprocess.check_output(["git", "describe", "--always", "--dirty"], text=True).strip()
TEST_DIR = REPO / "data" / "simulated" / "one_state_random_blocks_test"
OUT_DIR = REPO / "outputs" / "one_state" / "engines" / "posterior_random_blocks"

# The Prior and Schedule Design of decision 0005, from their stated 95% ranges.
z95 = stats.norm.ppf(0.975)
range_a, range_b = (0.75, 0.999), (0.01, 0.5)
range_sigma_epsilon, range_ratio, range_block_len = (1 / 15, 1 / 3), (1 / 15, 1 / 5), (10, 250)
prior = {
    "mu_logit_A": special.logit(range_a).mean(), "sigma_logit_A": np.ptp(special.logit(range_a)) / (2 * z95),
    "mu_logit_B": special.logit(range_b).mean(), "sigma_logit_B": np.ptp(special.logit(range_b)) / (2 * z95),
    "mu_log_sigma_epsilon": np.log(range_sigma_epsilon).mean(),
    "sigma_log_sigma_epsilon": np.ptp(np.log(range_sigma_epsilon)) / (2 * z95),
    "mu_log_ratio": np.log(range_ratio).mean(), "sigma_log_ratio": np.ptp(np.log(range_ratio)) / (2 * z95),
}
schedule_design = {
    "n_blocks_min": 5, "n_blocks_max": 16,
    "mu_log_block_len": np.log(range_block_len).mean(),
    "sigma_log_block_len": np.ptp(np.log(range_block_len)) / (2 * z95),
    "trial_type_concentration": 3.0,
}


# Training: Sittings are simulated as they are needed and never reused.
batch_size = 32  # Sittings per batch, all on one Schedule (BayesFlow's default batch size)
batches_per_epoch = 300
epochs = 100  # with batches_per_epoch: 960,000 Sittings in all
# Each new Sitting length makes JAX compile the training step again (about 5 s). Cutting
# every training Schedule down to a multiple of length_step Trials (at most 24 Trials
# lost, from the end) keeps the number of distinct lengths below about a hundred.
length_step = 25
n_validation = 300  # held-out Sittings on one Schedule, for the loss after each epoch

# Test Experiment: Schedules of the Schedule Design as they come (no cutting), several
# Sittings on each. 300 Sittings in all, as agreed for the Checks.
n_test_schedules = 30
n_sittings_per_test_schedule = 10

# Networks at BayesFlow's "Base" size, the size its guidance says to start with.
summary_dim = 12  # 3 summary numbers per parameter (BayesFlow's starting heuristic)
conv_filters = (32, 64)  # two convolution layers, each over a window of conv_window Trials
conv_window = 3
recurrent_units = 128  # size of the state the recurrent layers carry from Trial to Trial
flow_widths = (256, 256, 256, 256)
flow_time_embedding_dim = 32

n_posterior_draws = 1000  # per test Sitting, for the Checks (BayesFlow's default)

In [ ]:
# [rng] One stream for training and validation batches, one for the test Experiment; a seed for Keras
rng = np.random.default_rng(root_seed)
rng_batches, rng_test = rng.spawn(2)
keras.utils.set_random_seed(int(rng.integers(2**31)))
series_vars = mdl.TASK.CONDITION_VARS + mdl.TASK.OBSERVATION_VARS

## [simulator] Step 1: simulate a batch of Sittings on one Schedule

**Takes:** the Prior and the Schedule Design (from `#[config]`), and a number of
Sittings $n$.

**Does:** draws one Schedule; draws $n$ sets of parameter values from the Prior;
simulates one Sitting for each set, all on that Schedule. This is our own
`simulate_experiment`, asked for one Schedule with $n$ Sittings on it. Two
rearrangements follow. The Sittings are stacked into arrays, which is possible because
they share a Schedule and so have one length. And the Trials beyond a multiple of
`length_step` are cut off; a Sitting with its last few Trials removed is still a Sitting
of the Model, on the shortened Schedule, so nothing about the Model is approximated.

**Gives:** a dict of arrays. `p`, `v` and `y` have shape $(n, T)$: row $i$ is Sitting
$i$, column $t$ is Trial $t$; the rows of `p` and of `v` are all the same, because the
Schedule is shared. Each parameter has shape $(n, 1)$: row $i$ is the value that
produced Sitting $i$. $T$ changes from one call to the next.

**Why a function.** BayesFlow asks for training data one batch at a time, by calling a
function with the number of Sittings it wants. `sample_batch` is that function. BayesFlow
hands it no random generator, so it uses the notebook's `rng_batches` stream.

In [ ]:
# [sample-batch] One Schedule, n Sittings on it, as arrays (n, Trials); parameters as (n, 1)
def sample_batch(batch_shape):
    '''Simulate one batch for BayesFlow: Sittings that share one Schedule.

    Parameters
    ----------
    batch_shape : tuple of int, shape (1,)
        ``(n,)``, the number of Sittings wanted; this is how BayesFlow asks.

    Returns
    -------
    batch : dict of numpy.ndarray, float32
        One array of shape (n, T) per Condition and Observation of the Task (``p``,
        ``v``, ``y``), and one of shape (n, 1) per parameter of the Model. T is the
        length of the Schedule drawn for this batch, cut down to a multiple of
        `length_step` (left as it is if shorter than `length_step`).

    Uses and advances the notebook's `rng_batches`; reads `prior`, `schedule_design` and
    `length_step` from the config cell.
    '''
    n = batch_shape[0]
    sittings, ground_truth = experiments.simulate_experiment(
        rng_batches, mdl, prior, des.sample_schedule, schedule_design, n, n_sittings_per_schedule=n
    )
    n_trials = len(sittings[0]["y"])
    if n_trials >= length_step:
        n_trials -= n_trials % length_step
    batch = {name: np.stack([s[name][:n_trials] for s in sittings]).astype("float32") for name in series_vars}
    for name in mdl.PARAMETERS:
        batch[name] = np.array([[truth[name]] for truth in ground_truth], dtype="float32")
    return batch

`bf.make_simulator` wraps the function in the object BayesFlow expects: something with a
`sample(n)` method. `is_batched=True` says that our function makes all $n$ Sittings in
one call (otherwise BayesFlow would call it $n$ times and stack the results, giving every
Sitting its own Schedule and length).

The first use is the **validation batch**: `n_validation` Sittings on one Schedule,
simulated once and set aside. Training never sees them; the loss on them after each
epoch shows whether training is still improving.

In [ ]:
# [make-simulator] Wrap sample_batch for BayesFlow; simulate the validation batch and show its shapes
simulator = bf.make_simulator(sample_batch, is_batched=True)
validation = simulator.sample(n_validation)
pd.Series({name: values.shape for name, values in validation.items()}, name="shape in the validation batch")

## [route] Step 2: route the arrays to the two networks (the adapter)

**Takes:** a batch from step 1.

**Does:** three things, in the order written in the cell.

- `as_time_series` gives `p`, `v` and `y` a third axis, $(n, T) \to (n, T, 1)$, so that
  they can be laid side by side.
- `constrain` moves each parameter to an unbounded scale: $A$ and $B$, which lie between
  0 and 1, by the logit; the two noise sizes, which are positive, by a log-like
  transform. The inference network works on those scales, and BayesFlow undoes the
  transform on every draw, so no draw can fall outside a parameter's domain.
- `concatenate` lays the arrays side by side under the two names BayesFlow looks for.

**Gives:** `summary_variables`, shape $(n, T, 3)$: for each Sitting and Trial the three
numbers $(p_t, v_t, y_t)$. This is what the summary network reads. And
`inference_variables`, shape $(n, 4)$: the four transformed parameters. This is what the
inference network learns the distribution of.

The adapter is part of the Engine: the same routing is applied in training and whenever
the Engine is later given a Sitting.

In [ ]:
# [adapter] Route the arrays: p, v, y to the summary network; the parameters to the inference network
adapter = (
    bf.Adapter()
    .as_time_series(list(series_vars))
    .constrain(["A", "B"], lower=0, upper=1)
    .constrain(["sigma_eta", "sigma_epsilon"], lower=0)
    .concatenate(list(series_vars), into="summary_variables")
    .concatenate(list(mdl.PARAMETERS), into="inference_variables")
)
pd.Series({name: values.shape for name, values in adapter(validation).items()}, name="shape after the adapter")

## [engine] Step 3: build the Inference Engine

**Takes:** the sizes in `#[config]`, the adapter, and the simulator.

**Does:** creates the two networks, untrained, and the BayesFlow workflow that holds
them together with the adapter.

- **The summary network** is BayesFlow's `TimeSeriesNetwork`. In: $(n, T, 3)$, for any
  $T$. Out: $(n, 12)$, twelve learned summary numbers per Sitting. Inside, two
  convolution layers compute features of each Trial and its neighbours (5 Trials in
  all), and recurrent layers then read those features Trial by Trial, in both
  directions, carrying a state of 128 numbers.
- **The inference network** is flow matching, BayesFlow's recommended default. In
  training it sees the 12 summary numbers and the 4 parameters. Afterwards, given the 12
  summary numbers, it produces as many draws of the 4 parameters as asked for.

`standardize="inference_variables"` makes BayesFlow rescale the transformed parameters
to mean 0 and standard deviation 1. The Sittings are not rescaled; they are already in
units of the perturbation size.

**Gives:** `workflow`, the untrained Engine. It will save itself in `OUT_DIR` as
`engine.keras`.

In [ ]:
# [workflow] The two networks and the BayesFlow workflow that trains and saves them
OUT_DIR.mkdir(parents=True, exist_ok=True)
workflow = bf.BasicWorkflow(
    simulator=simulator,
    adapter=adapter,
    summary_network=bf.networks.TimeSeriesNetwork(
        summary_dim=summary_dim, filters=conv_filters, kernel_sizes=conv_window, recurrent_dim=recurrent_units
    ),
    inference_network=bf.networks.FlowMatching(
        subnet_kwargs={"widths": flow_widths, "time_embedding_dim": flow_time_embedding_dim}
    ),
    standardize="inference_variables",
    checkpoint_filepath=str(OUT_DIR),
    checkpoint_name="engine",
)

## [train] Step 4: train

**Takes:** the untrained Engine, the simulator of step 1, the validation batch.

**Does:** `epochs` $\times$ `batches_per_epoch` training steps. In each step BayesFlow
asks the simulator for a fresh batch of `batch_size` Sittings (one new Schedule), passes
it through the adapter and both networks, computes the loss, and adjusts the weights of
both networks a little to make the loss smaller. No Sitting is seen twice. An "epoch"
here is only a unit of bookkeeping: after every `batches_per_epoch` steps the Engine is
saved and the loss is computed on the validation batch.

The loss is that of flow matching (wiki:
[Concept-flow-matching](https://github.com/opherdonchin/MotorAdpatationSBI/wiki/Concept-flow-matching)).

**Gives:** the trained Engine, in memory as `workflow` and on disk as `engine.keras`;
and `history`, the training and validation loss after every epoch.

The first steps at each new Sitting length are slow, because JAX compiles the training
step for that length.

In [ ]:
# [fit] Train: fresh batches from the simulator; the Engine is saved after every epoch
history = workflow.fit_online(
    epochs=epochs, num_batches_per_epoch=batches_per_epoch, batch_size=batch_size,
    validation_data=validation, verbose=2,
)

In [ ]:
# [save-record] Save the loss history, and the record of what produced the Engine
_ = (OUT_DIR / "history.json").write_text(json.dumps(history.history))
_ = (OUT_DIR / "record.json").write_text(json.dumps({
    "model": mdl.__name__, "schedule_generator": f"{des.__name__}.sample_schedule",
    "prior": prior, "schedule_design": schedule_design,
    "summary_network": "TimeSeriesNetwork", "summary_dim": summary_dim, "conv_filters": conv_filters,
    "conv_window": conv_window, "recurrent_units": recurrent_units,
    "inference_network": "FlowMatching", "flow_widths": flow_widths,
    "epochs": epochs, "batches_per_epoch": batches_per_epoch, "batch_size": batch_size,
    "length_step": length_step, "root_seed": str(root_seed), "git_commit": GIT_COMMIT,
}, indent=2))

## [check-1] Step 5, Check 1: has training converged?

**Takes:** `history`. **Gives:** the two loss curves, and their averages over the last
10 epochs and the 10 before.

The loss is a squared error on a velocity, not a log probability, so it does not go to
zero. What to look for: both curves flat at the end, and the validation loss not above
the training loss. The validation batch is on a single Schedule, so its level need not
equal the training loss, which averages over Schedules; its trend is what matters.

In [ ]:
# [check-training] Training and validation loss per epoch
loss = pd.DataFrame(history.history)[["loss", "val_loss"]]
fig = bf.diagnostics.plots.loss(history)
pd.Series({
    "training loss, mean of last 10 epochs": round(loss["loss"].tail(10).mean(), 3),
    "training loss, mean of the 10 epochs before": round(loss["loss"].iloc[-20:-10].mean(), 3),
    "validation loss, mean of last 10 epochs": round(loss["val_loss"].tail(10).mean(), 3),
    "validation loss, mean of the 10 epochs before": round(loss["val_loss"].iloc[-20:-10].mean(), 3),
})

## [test] Step 6: make the test Experiment

**Takes:** the Prior and the Schedule Design; its own random stream, `rng_test`, so that
it does not depend on how long training ran.

**Does:** simulates a Simulated Experiment and saves it like any other Experiment, with
its Ground Truth apart. The Schedules come straight from the Schedule Design: their
lengths are not cut to the training grid, so most are lengths the Engine never trained
on. There are several Sittings on each Schedule.

**Gives:** on disk, `TEST_DIR` with the Sittings, the Ground Truth and a record.

In [ ]:
# [simulate-test] Simulate and save the test Experiment
n_test = n_test_schedules * n_sittings_per_test_schedule
sittings, ground_truth = experiments.simulate_experiment(
    rng_test, mdl, prior, des.sample_schedule, schedule_design, n_test,
    n_sittings_per_schedule=n_sittings_per_test_schedule,
)
experiments.save_experiment(TEST_DIR, sittings, ground_truth, {
    "model": mdl.__name__, "schedule_generator": f"{des.__name__}.sample_schedule",
    "prior": prior, "schedule_design": schedule_design,
    "n_sittings": n_test, "n_sittings_per_schedule": n_sittings_per_test_schedule,
    "root_seed": str(root_seed), "git_commit": GIT_COMMIT,
})
del sittings, ground_truth

The Experiment is then loaded from disk, the way an Analysis would load an Actual
Experiment. From here on the Engine sees only `test_sittings`: a list with one dict per
Sitting, each holding that Sitting's `p`, `v` and `y`, arrays of shape $(T,)$. The true
parameter values go into `test_truth`, shape $(300, 1)$ per parameter, which only the
Checks read.

In [ ]:
# [load-test] Load the test Sittings (for the Engine) and their true parameter values (for the Checks)
test_sittings = experiments.load_sittings(TEST_DIR)
truth, _ = experiments.load_ground_truth(TEST_DIR)
test_truth = {name: truth[name][:, None].astype("float32") for name in mdl.PARAMETERS}
test_lengths = np.array([len(s["y"]) for s in test_sittings])
pd.Series(test_lengths[::n_sittings_per_test_schedule]).describe().round(0).rename("Trials per test Schedule")

## [draw] Step 7: draw from the posterior of every test Sitting

**Takes:** the trained Engine and `test_sittings`.

**Does:** for each test Schedule, stacks its Sittings into arrays of shape
$(10, T)$, with no cutting, and calls `workflow.sample`. For each Sitting the adapter
routes `p`, `v`, `y` as in step 2, the summary network gives its 12 numbers, and the
inference network turns them into `n_posterior_draws` draws of the four parameters, which
BayesFlow moves back to their own scales. The Engine is not told the true values.

**Gives:** `draws`, a dict with one array per parameter, of shape
(Sittings, draws, 1) $= (300, 1000, 1)$. `draws["B"][i, :, 0]` is the posterior of $B$
for Sitting $i$, as 1,000 numbers.

(The cell prints two progress bars per Schedule.)

In [ ]:
# [sample-test] Posterior draws for every test Sitting, one Schedule at a time; shapes of the result
draws_by_schedule = []
for first in range(0, n_test, n_sittings_per_test_schedule):
    group = test_sittings[first : first + n_sittings_per_test_schedule]
    conditions = {name: np.stack([s[name] for s in group]).astype("float32") for name in series_vars}
    draws_by_schedule.append(workflow.sample(conditions=conditions, num_samples=n_posterior_draws))
draws = {name: np.concatenate([d[name] for d in draws_by_schedule]) for name in mdl.PARAMETERS}
pd.Series({name: values.shape for name, values in draws.items()}, name="shape of the draws")

## [check-2] Step 8, Checks 2 and 3: calibration, recovery and contraction

**Takes:** `draws` and `test_truth`. **Gives:** BayesFlow's own diagnostic figures and
table, shown here and saved in `OUT_DIR`.

- **Calibration** (`calibration_ecdf`; *Calibration Error* and *Log Gamma* in the
  table): for each parameter, where the true value ranks among the posterior draws. If
  the posterior is honest, the ranks are uniform and the curve stays inside the grey
  band. *Log Gamma* is a test of that: below 0 means uniform ranks are rejected at the
  5% level.
- **Recovery** (`recovery`, and *NRMSE*): posterior median against true value.
- **Contraction** (`z_score_contraction`, and *Posterior Contraction*): 1 minus the
  posterior's variance over the Prior's; 0 means the Sitting taught nothing about the
  parameter, 1 means it pinned it down.

In [ ]:
# [check-diagnostics] BayesFlow's default diagnostics: figures and table, both saved
figures = workflow.plot_default_diagnostics(test_data=test_truth, samples=draws)
for name, fig in figures.items():
    fig.savefig(OUT_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
metrics = workflow.compute_default_diagnostics(test_data=test_truth, samples=draws, as_data_frame=True)
metrics.to_csv(OUT_DIR / "metrics.csv")
metrics.round(3)

## [offset] Step 8, continued: how far off, and in which direction

**Takes:** `draws` and `test_truth`. **Gives:** one small table.

The calibration Check says whether the posterior is honest, not how large a failure is.
Two plain numbers per parameter give the size. The *mean offset* is the posterior mean
minus the true value, in units of the posterior's own standard deviation, averaged over
the test Sittings: 0 if the posterior is centred, negative if it sits too low. The
*coverage* is how often the true value lies inside the posterior's central 50% and 90%
intervals: more often than 50% and 90% means the posterior is too wide.

In [ ]:
# [check-offset] Mean offset of the posterior from the truth, and coverage of its central intervals
offset = {}
for name in mdl.PARAMETERS:
    d, t = draws[name][..., 0], test_truth[name]
    rank = (d < t).mean(axis=1)
    offset[name] = {
        "mean offset (posterior sd)": ((d.mean(axis=1) - t[:, 0]) / d.std(axis=1)).mean(),
        "truth inside 50% interval": (np.abs(rank - 0.5) < 0.25).mean(),
        "truth inside 90% interval": (np.abs(rank - 0.5) < 0.45).mean(),
    }
pd.DataFrame(offset).round(3)

## [check-4] Step 8, Check 4: does Sitting length matter?

**Takes:** `draws`, `test_truth` and the length of each test Sitting. **Gives:** the
table of Checks 2 and 3, twice.

The same table for the test Sittings on the shorter and on the longer Schedules. Longer
Sittings hold more information, so better recovery and more contraction are expected
there. What would be a warning is *calibration* that differs between the two: it would
mean the Engine handles some lengths badly.

In [ ]:
# [check-length] Default diagnostics for the Sittings on the shorter and on the longer Schedules
is_long = test_lengths >= np.median(test_lengths)
halves = {"shorter": ~is_long, "longer": is_long}
by_length = pd.concat({
    f"{label} ({test_lengths[rows].min()} to {test_lengths[rows].max()} Trials, {rows.sum()} Sittings)":
        workflow.compute_default_diagnostics(
            test_data={name: values[rows] for name, values in test_truth.items()},
            samples={name: values[rows] for name, values in draws.items()},
            as_data_frame=True,
        )
    for label, rows in halves.items()
})
by_length.round(3)

## [use] Step 9: using the Engine on one Sitting

What the Engine gives for a single Sitting: draws from the joint posterior of the four
parameters. Here, the first test Sitting, with its true values in red. The same call,
`workflow.sample`, is what an Analysis of an Actual Sitting would use.

In [ ]:
# [example] Posterior draws for the first test Sitting, with the true values
example = 0
grid = bf.diagnostics.plots.pairs_posterior(
    estimates={name: values[example] for name, values in draws.items()},
    targets={name: values[example] for name, values in test_truth.items()},
)

## [results] What this shows

*To be written from the executed notebook.*